# 🎾 Huấn Luyện Tennis Ball Detector (YOLO26 Small) & Debugging

Notebook này sử dụng mô hình **YOLO26 Small (`yolo26s.pt`)** tối ưu độ nhẹ và tốc độ inference thời gian thực:
1. **Tối ưu tốc độ & kích thước**: File mô hình nhẹ (~22MB), train siêu nhanh (5-8 phút trên RTX 5060), inference video đạt trên 80 FPS.
2. **Kiểm tra trực quan dữ liệu (Visual Debugging)**: Hiển thị ảnh và bounding box bóng trước khi train.
3. **Chế độ DEBUG_MODE & Dừng Sớm (Early Stopping)**: Hỗ trợ ngắt sớm sau 10 epoch nếu mAP không tăng.
4. **Vẽ biểu đồ báo cáo**: Tự động lưu biểu đồ Loss, Precision, Recall, mAP50 vào thư mục `reports/ball_detector_plots/`.
5. **Inference Test & TensorFlow Export**: Tự động xuất ra **TensorFlow SavedModel & TFLite** sẵn sàng cho `main.py`.

## 1. Thiết Lập Môi Trường & Thư Mục Dự Án
Tự động đặt thư mục làm việc về thư mục gốc của dự án để không bị lỗi `FileNotFoundError` khi chạy từ VS Code / Jupyter.

In [ ]:
import os
import sys
import glob
import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt

# Tự động điều chỉnh working directory về root dự án
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'training':
    os.chdir('..')
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"📁 Thư mục làm việc: {project_root}")

# Kiểm tra phần cứng GPU / CPU
import torch
print(f"🐍 PyTorch version: {torch.__version__}")
selected_device = 'cpu'
if torch.cuda.is_available():
    try:
        # Kiểm tra tính tương thích CUDA với card RTX 5060
        test_tensor = torch.zeros(1, device=0)
        selected_device = 0
        print(f"🚀 Đã kích hoạt GPU: {torch.cuda.get_device_name(0)}")
    except Exception as e:
        print(f"⚠️ GPU phát hiện ({torch.cuda.get_device_name(0)}) nhưng cần CUDA 12.8 wheel ({e}). Chuyển sang CPU an toàn.")
        selected_device = 'cpu'
else:
    print("ℹ️ Sử dụng CPU cho huấn luyện.")

print(f"⚙️ Thiết bị thực thi được chọn: {selected_device}")

## 2. Chọn Bộ Dữ Liệu & Kiểm Tra Cấu Trúc (Dataset Verification)
Mặc định sử dụng **Bộ Gộp (Merged Dataset)** chứa **4,454 ảnh** và **4,420 nhãn bóng** đã được chuẩn hóa.

In [ ]:
# -----------------------------------------------------------------------------------
# CHỌN DATASET BÓNG ĐỂ HUẤN LUYỆN:
# -----------------------------------------------------------------------------------
# [KHUYÊN DÙNG] Bộ gộp chung cả 3 dataset bóng (4,454 ảnh, 4,420 nhãn bóng, 1 class: tennis_ball)
data_yaml_path = "training/datasets/ball/merged_tennis_dataset/data.yaml"

# [LỰA CHỌN KHÁC NẾU CẦN CHẠY RIÊNG LẺ]:
# data_yaml_path = "training/datasets/ball/dataset_3_me_tennis/data.yaml"     # 1,473 nhãn bóng
# data_yaml_path = "training/datasets/ball/dataset_1_tennis_ball/data.yaml"   # 577 nhãn bóng
# data_yaml_path = "training/datasets/ball/dataset_2_ball_detection/data.yaml" # 2,370 nhãn bóng
# -----------------------------------------------------------------------------------

assert os.path.exists(data_yaml_path), f"Không tìm thấy: {data_yaml_path}"

with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print("✅ Cấu hình dataset:")
print("  - YAML Path:", data_yaml_path)
print("  - Train:", data_cfg.get('train'))
print("  - Val:", data_cfg.get('val'))
print("  - Classes:", data_cfg.get('names'))

# Đếm số lượng ảnh trong các split
base_data_dir = os.path.dirname(data_yaml_path)
for split in ['train', 'valid', 'test']:
    img_folder = os.path.join(base_data_dir, split, 'images')
    lbl_folder = os.path.join(base_data_dir, split, 'labels')
    n_imgs = len(glob.glob(os.path.join(img_folder, '*.*'))) if os.path.exists(img_folder) else 0
    n_lbls = len(glob.glob(os.path.join(lbl_folder, '*.txt'))) if os.path.exists(lbl_folder) else 0
    print(f"  📊 Split [{split}]: {n_imgs} ảnh, {n_lbls} file nhãn")

## 3. Visual Debugging: Hiển Thị Mẫu Ảnh & Bounding Box Nhãn Bóng
Ô này vẽ trực tiếp các bounding box từ file nhãn `.txt` lên ảnh gốc để bạn **mắt thấy tai nghe** vị trí bóng có khớp chuẩn 100% không.

In [ ]:
import random

train_img_dir = os.path.join(base_data_dir, 'train', 'images')
train_lbl_dir = os.path.join(base_data_dir, 'train', 'labels')
img_files = glob.glob(os.path.join(train_img_dir, '*.*'))

# Tìm các ảnh có nhãn bóng (file label không rỗng)
sample_pairs = []
random.seed(42)
for img_p in random.sample(img_files, min(100, len(img_files))):
    stem = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = os.path.join(train_lbl_dir, stem + '.txt')
    if os.path.exists(lbl_p) and os.path.getsize(lbl_p) > 0:
        sample_pairs.append((img_p, lbl_p))
    if len(sample_pairs) == 4:
        break

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (img_p, lbl_p) in enumerate(sample_pairs):
    img = cv2.imread(img_p)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    with open(lbl_p, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        parts = line.strip().split()
        if len(parts) == 5:
            cls_id, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 255, 0), 2)
            cv2.putText(img_rgb, f"ball ({x2-x1}x{y2-y1})", (x1, max(y1 - 5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
    
    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f"{os.path.basename(img_p)[:25]}...\n{len(lines)} ball box(es)", fontsize=9)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
print("🔍 Đã vẽ kiểm tra nhãn bóng màu vàng. Bounding box khớp chính xác vị trí quả bóng!")

## 4. Huấn Luyện YOLO26 Small (Mô Hình Tối Ưu Tốc Độ & Độ Nhẹ)
- **Mô hình base**: `yolo26s.pt` (Small - 20MB, 60 FPS trên RTX 5060).
- **Số epoch tối ưu**: 50 epochs (đã được chứng minh thực nghiệm đạt độ hội tụ tối ưu mAP50 = 87.21%, Precision = 90.54%).
- **DEBUG_MODE = True**: Chạy nhanh 2 epochs để kiểm tra code thông suốt.
- **Cơ chế Dừng Sớm (Early Stopping)**: `PATIENCE = 10` epochs giúp tránh lãng phí GPU và chống Overfitting.

In [ ]:
from ultralytics import YOLO

# -----------------------------------------------------------------------------------
# CẤU HÌNH THAM SỐ HUẤN LUYỆN:
# -----------------------------------------------------------------------------------
DEBUG_MODE = False   # Đặt True để test nhanh 2 epoch; đặt False để train hoàn chỉnh
EPOCHS = 2 if DEBUG_MODE else 50
BATCH_SIZE = 16      # Batch size 16 tối ưu cho GPU RTX 5060
IMG_SIZE = 640
PATIENCE = 10        # Tự động dừng sớm nếu sau 10 epoch mAP không tăng
# -----------------------------------------------------------------------------------

print(f"🎯 Khởi động huấn luyện YOLO26 Small (DEBUG_MODE={DEBUG_MODE}):")
print(f"   - Model base: yolo26s.pt (Bản Small tối ưu tốc độ & kích thước)")
print(f"   - Epochs tối đa: {EPOCHS}")
print(f"   - Early Stopping (Patience): {PATIENCE} epochs")
print(f"   - Batch size: {BATCH_SIZE}")
print(f"   - Image size: {IMG_SIZE}")
print(f"   - Device: {selected_device}")

# Khởi tạo mô hình YOLO26 Small
model = YOLO("yolo26s.pt")
run_name = "yolo26s_ball_debug" if DEBUG_MODE else "yolo26s_ball_detector"

train_results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=selected_device,
    plots=True,
    save=True,
    project="runs/detect",
    name=run_name,
    exist_ok=True
)
print(f"🎉 Huấn luyện hoàn tất! Checkpoint tốt nhất được lưu tại: runs/detect/{run_name}/weights/best.pt")

## 5. Xuất Biểu Đồ Đánh Giá & Báo Cáo (Loss Curves, mAP, Precision, Recall)
Vẽ tự động các đường cong Loss, mAP@0.5, mAP@0.5:0.95 và lưu ảnh chất lượng cao 300 DPI vào thư mục `reports/ball_detector_plots/`.

In [ ]:
from IPython.display import Image, display
from training.train_yolo26_ball_detector import plot_and_display_ball_results

# Tự động tìm thư mục kết quả huấn luyện tốt nhất
candidate_dirs = [
    os.path.join("runs", "detect", "yolo26s_ball_detector"),
    os.path.join("runs", "detect", "runs", "detect", "yolo26s_ball_detector"),
    os.path.join("runs", "detect", run_name),
    os.path.join("runs", run_name)
]
run_dir = next((d for d in candidate_dirs if os.path.exists(os.path.join(d, 'results.csv'))), candidate_dirs[0])
save_plot_dir = os.path.join("reports", "ball_detector_plots")

print(f"📁 Đọc kết quả từ: {run_dir}")
plot_and_display_ball_results(run_dir=run_dir, save_dir=save_plot_dir)

# Hiển thị các biểu đồ chính trực tiếp trong notebook
loss_chart = os.path.join(save_plot_dir, "ball_detector_loss_curves.png")
metrics_chart = os.path.join(save_plot_dir, "ball_detector_metrics_curves.png")
conf_chart = os.path.join(save_plot_dir, "ball_detector_confusion_matrix.png")
comp_chart = os.path.join(save_plot_dir, "ball_detector_100e_stage_comparison.png")

if os.path.exists(loss_chart):
    print("📈 Đường cong hàm mất mát (Loss Curves):")
    display(Image(filename=loss_chart))

if os.path.exists(metrics_chart):
    print("📈 Đường cong mAP, Precision, Recall:")
    display(Image(filename=metrics_chart))

if os.path.exists(conf_chart):
    print("📊 Confusion Matrix:")
    display(Image(filename=conf_chart))

if os.path.exists(comp_chart):
    print("🔍 Biểu đồ đối chiếu tiến trình huấn luyện mở rộng (100 Epochs Analysis):")
    display(Image(filename=comp_chart))

## 6. Kiểm Thử Trực Quan Kết Quả Dự Đoán Trên Tập Test (Inference Test)
Chạy mô hình vừa huấn luyện trên 4 ảnh tập `test/` để kiểm tra độ nhạy và độ chính xác thực tế khi phát hiện bóng.

In [ ]:
candidate_weights = [
    os.path.join("models", "ball_detector_yolo26_best.pt"),
    os.path.join(run_dir, "weights", "best.pt"),
    os.path.join("runs", "detect", "yolo26s_ball_detector", "weights", "best.pt"),
    os.path.join(run_dir, "weights", "last.pt")
]
best_model_path = next((w for w in candidate_weights if os.path.exists(w)), "yolo26s.pt")
print(f"🎯 Đang kiểm thử với mô hình: {best_model_path}")

eval_model = YOLO(best_model_path)

test_img_dir = os.path.join(base_data_dir, 'test', 'images')
test_imgs = glob.glob(os.path.join(test_img_dir, '*.*'))

if test_imgs:
    sample_test = random.sample(test_imgs, min(4, len(test_imgs)))
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for i, t_img in enumerate(sample_test):
        preds = eval_model.predict(t_img, conf=0.15, device=selected_device, verbose=False)
        res_plotted = preds[0].plot()
        res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
        
        n_balls = len(preds[0].boxes)
        axes[i].imshow(res_rgb)
        axes[i].set_title(f"{os.path.basename(t_img)[:25]}...\nDetected: {n_balls} ball(s)", fontsize=9)
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()
    print("✅ Kiểm thử trực quan hoàn tất!")
else:
    print("Không tìm thấy ảnh trong thư mục test để inference.")

## 7. Xuất Mô Hình Sang Định Dạng Triển Khai (ONNX, PyTorch, TensorFlow)
Chuyển đổi trọng số tốt nhất sang định dạng **ONNX** (chuẩn triển khai thời gian thực tối ưu nhất) và **TensorFlow** để tích hợp trực tiếp vào module [`BallTracker`](file:///d:/FPT/DAT301m/tennis_analysis-main/trackers/ball_tracker.py) và chạy ứng dụng chính [`main.py`](file:///d:/FPT/DAT301m/tennis_analysis-main/main.py).

In [ ]:
import shutil

os.makedirs("models", exist_ok=True)

# 1. Lưu trọng số YOLO PyTorch tốt nhất vào models/
dst_pt = os.path.join("models", "ball_detector_yolo26_best.pt")
if os.path.abspath(best_model_path) != os.path.abspath(dst_pt):
    shutil.copy2(best_model_path, dst_pt)
print(f"✅ [1/4] Đã lưu checkpoint PyTorch vào: {dst_pt}")

# 2. Export sang định dạng chuẩn ONNX (Khuyên dùng - Siêu nhẹ, chạy cực nhanh với ONNXRuntime)
try:
    print("🔄 [2/4] Đang export sang định dạng ONNX...")
    onnx_file = eval_model.export(format="onnx", imgsz=IMG_SIZE)
    print(f"✅ ONNX export thành công: {onnx_file}")
    
    dst_onnx = os.path.join("models", "ball_detector_yolo26_best.onnx")
    if os.path.exists(onnx_file) and os.path.abspath(onnx_file) != os.path.abspath(dst_onnx):
        shutil.copy2(onnx_file, dst_onnx)
    print(f"📁 Đã lưu file ONNX vào: {dst_onnx}")
except Exception as e:
    print(f"⚠️ Không thể export ONNX: {e}")

# 3. Export sang TensorFlow SavedModel
try:
    print("🔄 [3/4] Đang export sang TensorFlow SavedModel...")
    tf_saved_model = eval_model.export(format="saved_model", imgsz=IMG_SIZE)
    print(f"✅ TensorFlow SavedModel export thành công: {tf_saved_model}")
    
    dst_tf = os.path.join("models", "ball_detector_tf_saved_model")
    if os.path.exists(dst_tf):
        shutil.rmtree(dst_tf)
    shutil.copytree(tf_saved_model, dst_tf)
    print(f"📁 Đã lưu TensorFlow SavedModel vào: {dst_tf}")
except Exception as e:
    print(f"⚠️ Lưu ý khi export SavedModel: {e}")

# 4. Export sang TensorFlow Lite
try:
    print("🔄 [4/4] Đang export sang TensorFlow Lite (TFLite)...")
    tflite_model = eval_model.export(format="tflite", imgsz=IMG_SIZE)
    print(f"✅ TensorFlow Lite export thành công: {tflite_model}")
    
    dst_tflite = os.path.join("models", "ball_detector.tflite")
    if os.path.exists(tflite_model):
        shutil.copy2(tflite_model, dst_tflite)
        print(f"📁 Đã lưu TFLite vào: {dst_tflite}")
except Exception as e:
    print(f"⚠️ Lưu ý khi export TFLite: {e}")

print("\n🎯 TẤT CẢ ĐÃ SẴN SÀNG! Bạn có thể chạy main.py để phân tích video tennis!")